# Advanced Problems with Solutions: Function Arguments and Mutability

This notebook contains advanced, interview-style problems on **function arguments**, **object identity**, **mutability**, **aliasing**, and **copying semantics** in Python.

## Best practices emphasized
- Distinguish **rebinding a local name** from **mutating a shared object**.
- Avoid **mutable default arguments** unless you explicitly want shared state.
- Use **shallow copy** vs **deep copy** intentionally.
- Document whether a function **mutates its inputs**.
- Prefer returning new values when side effects would be surprising.


## Problem 1 — Rebinding vs Mutation

Consider the following function:

1. Predict the final values of `a` and `b`.
2. Predict whether the `id` of `x` inside the function changes.
3. Explain why the behavior differs between the first and second assignment.


In [1]:
def transform(x):
    print(f"start: id(x)={hex(id(x))}, x={x}")
    x.append(99)
    print(f"after append: id(x)={hex(id(x))}, x={x}")
    x = x + [100]
    print(f"after x = x + [100]: id(x)={hex(id(x))}, x={x}")
    return x

a = [1, 2]
b = transform(a)
print(f"outside: id(a)={hex(id(a))}, a={a}")
print(f"outside: id(b)={hex(id(b))}, b={b}")

start: id(x)=0x2698ce69dc0, x=[1, 2]
after append: id(x)=0x2698ce69dc0, x=[1, 2, 99]
after x = x + [100]: id(x)=0x2698ce6af00, x=[1, 2, 99, 100]
outside: id(a)=0x2698ce69dc0, a=[1, 2, 99]
outside: id(b)=0x2698ce6af00, b=[1, 2, 99, 100]


### Solution 1

`x.append(99)` **mutates** the original list object, so both `x` and `a` still refer to the same list after that operation.

`x = x + [100]` creates a **new list** and rebinds the local name `x` to that new object. That rebinding does **not** affect the caller's variable `a`.

Therefore:
- `a` becomes `[1, 2, 99]`
- `b` becomes `[1, 2, 99, 100]`
- the `id` stays the same after `append`, but changes after `x = x + [100]`


## Problem 2 — The Mutable Default Argument Trap

The function below is intended to build a running history.

1. Predict the outputs.
2. Identify the bug.
3. Rewrite the function using best practices.


In [2]:
def record(value, history=[]):
    history.append(value)
    return history

print(record('a'))
print(record('b'))
print(record('c', ['fresh']))
print(record('d'))

['a']
['a', 'b']
['fresh', 'c']
['a', 'b', 'd']


### Solution 2

Default argument values are evaluated **once**, when the function is defined, not each time it is called.

That means the same default list is reused across calls where `history` is not provided.

Expected output:
- `['a']`
- `['a', 'b']`
- `['fresh', 'c']`
- `['a', 'b', 'd']`

A best-practice rewrite uses `None` as the sentinel default.

In [3]:
def record_fixed(value, history=None):
    if history is None:
        history = []
    history.append(value)
    return history

print(record_fixed('a'))
print(record_fixed('b'))
print(record_fixed('c', ['fresh']))
print(record_fixed('d'))

['a']
['b']
['fresh', 'c']
['d']


## Problem 3 — Strings, Lists, and `+=`

The operator `+=` can behave differently depending on the object type.

For each call below:
1. Predict whether the object passed by the caller is changed.
2. Predict whether the `id` stays the same inside the function.
3. Explain the difference between immutable and mutable types here.


In [4]:
def add_world(text):
    print(f"before: id(text)={hex(id(text))}, text={text!r}")
    text += ' world'
    print(f"after:  id(text)={hex(id(text))}, text={text!r}")
    return text

def add_item(items):
    print(f"before: id(items)={hex(id(items))}, items={items}")
    items += [10]
    print(f"after:  id(items)={hex(id(items))}, items={items}")
    return items

s = 'hello'
lst = [1, 2, 3]

s_result = add_world(s)
lst_result = add_item(lst)

print('outside string:', s, s_result)
print('outside list:', lst, lst_result)

before: id(text)=0x2698cab0390, text='hello'
after:  id(text)=0x2698ce92ab0, text='hello world'
before: id(items)=0x2698ce698c0, items=[1, 2, 3]
after:  id(items)=0x2698ce698c0, items=[1, 2, 3, 10]
outside string: hello hello world
outside list: [1, 2, 3, 10] [1, 2, 3, 10]


### Solution 3

For strings:
- strings are **immutable**
- `text += ' world'` creates a **new string**
- the caller's `s` is unchanged

For lists:
- lists are **mutable**
- `items += [10]` usually performs an **in-place mutation** (similar to `extend`)
- the caller's `lst` is changed

This is a classic reminder that identical syntax can trigger different object-level behavior depending on the type.

## Problem 4 — Immutable Container, Mutable Contents

A student claims: “Tuples are immutable, so nothing inside them can change.”

Use the code below to determine:
1. Which objects change and which do not.
2. Whether the tuple object itself is replaced.
3. Why this does not violate tuple immutability.


In [5]:
def update(config):
    print(f"tuple id before = {hex(id(config))}")
    print(f"inner list id before = {hex(id(config[1]))}")
    config[1].append('debug')
    print(f"tuple id after  = {hex(id(config))}")
    print(f"inner list id after  = {hex(id(config[1]))}")

settings = ('v1', ['safe', 'fast'])
update(settings)
print(settings)

tuple id before = 0x2698ce91d80
inner list id before = 0x2698ce69ec0
tuple id after  = 0x2698ce91d80
inner list id after  = 0x2698ce69ec0
('v1', ['safe', 'fast', 'debug'])


### Solution 4

The tuple object is unchanged: its identity stays the same, and its element references stay in the same positions.

However, the **list object stored as one of the tuple's elements** is mutable, so its internal state can change.

Tuple immutability means:
- you cannot reassign `config[1]` to a different object
- but if `config[1]` already refers to a mutable object, that object may still be mutated

So the tuple is immutable, but the objects it references do not have to be.

## Problem 5 — Shallow Copy vs Deep Copy

Study the nested data structure below.

1. Predict the values of `original`, `shallow`, and `deep` after mutation.
2. Explain exactly which nested objects are shared.
3. State when shallow copy is safe and when deep copy is necessary.


In [6]:
from copy import copy, deepcopy

original = {
    'name': 'report',
    'tags': ['draft', 'internal'],
    'meta': {'pages': 3}
}

shallow = copy(original)
deep = deepcopy(original)

original['tags'].append('review')
original['meta']['pages'] = 4
original['name'] = 'final-report'

print('original =', original)
print('shallow  =', shallow)
print('deep     =', deep)

print('shared tags with shallow:', original['tags'] is shallow['tags'])
print('shared meta with shallow:', original['meta'] is shallow['meta'])
print('shared tags with deep:', original['tags'] is deep['tags'])
print('shared meta with deep:', original['meta'] is deep['meta'])

original = {'name': 'final-report', 'tags': ['draft', 'internal', 'review'], 'meta': {'pages': 4}}
shallow  = {'name': 'report', 'tags': ['draft', 'internal', 'review'], 'meta': {'pages': 4}}
deep     = {'name': 'report', 'tags': ['draft', 'internal'], 'meta': {'pages': 3}}
shared tags with shallow: True
shared meta with shallow: True
shared tags with deep: False
shared meta with deep: False


### Solution 5

A **shallow copy** duplicates only the outer container. Nested objects are still shared.

So after mutation:
- `shallow['tags']` sees `'review'`
- `shallow['meta']['pages']` becomes `4`
- but `shallow['name']` remains `'report'` because rebinding `original['name']` updates only the outer dictionary entry

A **deep copy** recursively copies nested structures, so `deep` remains fully independent.

Use shallow copy when nested sharing is acceptable or intended. Use deep copy when you need full independence of nested mutable state.

## Problem 6 — Debugging an Unexpected Side Effect

The function below is supposed to return a sorted version of a list of scores while preserving the original input.

1. Identify the bug.
2. Explain why the caller sees unexpected behavior.
3. Provide two correct fixes.


In [7]:
def top_three(scores):
    scores.sort(reverse=True)
    return scores[:3]

values = [50, 80, 70, 90]
best = top_three(values)

print('best   =', best)
print('values =', values)

best   = [90, 80, 70]
values = [90, 80, 70, 50]


### Solution 6

The bug is that `list.sort()` mutates the original list in place.

Because `scores` and `values` refer to the same list object, sorting `scores` also changes `values`.

Two correct fixes are:
1. use `sorted(scores, reverse=True)` to create a new list
2. make a copy first, then sort the copy


In [8]:
def top_three_fixed_v1(scores):
    return sorted(scores, reverse=True)[:3]

def top_three_fixed_v2(scores):
    copied = scores.copy()
    copied.sort(reverse=True)
    return copied[:3]

values1 = [50, 80, 70, 90]
values2 = [50, 80, 70, 90]

print(top_three_fixed_v1(values1), values1)
print(top_three_fixed_v2(values2), values2)

[90, 80, 70] [50, 80, 70, 90]
[90, 80, 70] [50, 80, 70, 90]


## Problem 7 — Designing a Safe API

Write a function `normalize_names(names)` that:
- accepts a list of strings
- strips whitespace
- converts names to title case
- removes empty entries
- **does not mutate the caller's list**

Then explain why your implementation is safer than an in-place version for library code.


In [9]:
def normalize_names(names):
    return [name.strip().title() for name in names if name.strip()]

raw = ['  alice', 'BOB  ', '   ', 'charLie']
clean = normalize_names(raw)

print('raw   =', raw)
print('clean =', clean)
print('same object?', raw is clean)

raw   = ['  alice', 'BOB  ', '   ', 'charLie']
clean = ['Alice', 'Bob', 'Charlie']
same object? False


### Solution 7

This solution builds and returns a **new list**, leaving the input untouched.

That is safer for API design because callers are less likely to experience surprising side effects. In-place mutation can be efficient, but it should be explicit and well documented.

A strong convention is:
- functions with side effects should make that obvious
- functions that transform data should often return new values instead


## Problem 8 — Mixed Mutability in a Realistic Pipeline

Analyze the function below.

1. Predict the final state of `payload`.
2. Predict the returned object.
3. Explain which operations mutate shared state and which only rebind local names.


In [10]:
def process_payload(data):
    data['items'].append('X')
    data['status'] = data['status'] + '-processed'
    data = {
        **data,
        'items': data['items'] + ['Y']
    }
    return data

payload = {
    'status': 'new',
    'items': ['A', 'B']
}

result = process_payload(payload)
print('payload =', payload)
print('result  =', result)
print('same dict object?', payload is result)
print('same items object?', payload['items'] is result['items'])

payload = {'status': 'new-processed', 'items': ['A', 'B', 'X']}
result  = {'status': 'new-processed', 'items': ['A', 'B', 'X', 'Y']}
same dict object? False
same items object? False


### Solution 8

Step by step:
- `data['items'].append('X')` mutates the original shared list
- `data['status'] = data['status'] + '-processed'` updates the original dictionary entry with a newly created string
- then `data = {...}` rebinds the local name `data` to a **new dictionary**
- within that new dictionary, `'items': data['items'] + ['Y']` creates a **new list**

So:
- `payload` becomes `{'status': 'new-processed', 'items': ['A', 'B', 'X']}`
- `result` becomes `{'status': 'new-processed', 'items': ['A', 'B', 'X', 'Y']}`
- `payload is result` is `False`
- `payload['items'] is result['items']` is `False`

This example is a good reminder that a single function can mix mutation and rebinding in ways that are hard to reason about. In production code, prefer a clearer style.

## Summary

Core rule:

> A function receives references to objects. Whether the caller observes a change depends on whether the function **mutates the object** or merely **rebinds its local parameter name**.

Keep these questions in mind:
1. Is the object mutable?
2. Is the operation mutating the object or creating a new one?
3. Are nested mutable objects shared?
4. Does the function contract clearly document side effects?
